# 12 — VSL400 · keypoint MediaPipe (Kaggle) · Graph Encoder + Spatial/Temporal Transformer · 70 từ

Notebook **độc lập**, chạy từ đầu đến cuối nhánh **pose** trong thiết kế RGB–pose của dự án. Chỉ dùng góc **front**:

```
Kaggle keypoints_splited/*.npy [T, 76, 3]  (MediaPipe Holistic, người đăng đã trích sẵn)
   │  bỏ 8 điểm chân (ngoài khung crop) → 68 điểm: 25 thân trên + cổ + 2 × 21 điểm tay
   │  nội suy khoảng trống ≤ 3 frame · lấy mẫu đều 64 frame · x,y,z + vận tốc + vector xương
   ▼
[B, 64, 68, 9] → Graph Encoder (2 graph block) → Spatial Transformer (2 layer · 4 head)
              → Joint Pooling → Temporal Transformer (3 layer · 4 head) → Pose embedding [B, 256]
              → bộ phân loại 70 từ
```

Các bước:

1. **Manifest + split toàn bộ 400 từ.** Metadata 7 phần của VSL400 được đọc thẳng từ file zip Kaggle (không tải cả file). Split chia theo **người ký**, tỉ lệ 22 / 3 / 3 (seed 42, luôn cho cùng một kết quả). Nếu notebook 11 đã chạy, kết quả của nó được dùng lại.
2. **Chọn 70 từ** chỉ theo **số lần ký trong tập train**, và chỉ giữ từ có mặt ở cả 3 split. Split không bị chia lại.
3. **Tải đúng keypoint của 70 từ** (bản *canonical*, không dùng bản augmented) và ghép theo mã video + tên từ. Toàn bộ được đóng thành một file `.npz` lưu trên Drive.
4. **Train** với dropout, weight decay, label smoothing và tăng cường dữ liệu (cắt thời gian, xoay/co giãn nhẹ). Early stopping theo **validation loss**; checkpoint được lưu trên Drive sau mỗi epoch và có thể resume.
5. **Đánh giá:** top-1, top-5, macro-F1, F1 từng từ, ma trận nhầm lẫn. Tập **test chỉ chạy khi bật `RUN_TEST`**, và chỉ nên bật một lần, sau khi đã chốt cấu hình.

**Nguồn dữ liệu:** bộ Kaggle [`nguyenanfms/vsl-vietnamese-sign-language-v2`](https://www.kaggle.com/datasets/nguyenanfms/vsl-vietnamese-sign-language-v2) (phiên bản 8) là **bản đăng lại của bên thứ ba** từ VSL400; keypoint do người đăng tự trích bằng [code công khai](https://github.com/nguyenanfms/VSL-VietnameseSignLanguage). Tác giả VSL400 hiện yêu cầu ký Data Usage Agreement. Hãy xác nhận quyền sử dụng với tác giả (matej.sindelar@vsb.cz) rồi mới đặt `CONFIRM_KAGGLE_VSL400_PERMISSION = True`.

Trước khi chạy:
- Chọn **Runtime → Change runtime type → T4 GPU**.
- Tạo Colab Secrets `KAGGLE_USERNAME` và `KAGGLE_KEY` (kaggle.com → Settings → API → Create New Token), và bật *Notebook access* cho cả hai.

In [ ]:
PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/vsl400-mediapipe-pose-transformer'  # @param {type:'string'}
KAGGLE_DATASET = 'nguyenanfms/vsl-vietnamese-sign-language-v2'
KAGGLE_VERSION = 8  # @param {type:'integer'}
# Chỉ bật sau khi đã xác nhận với tác giả VSL400 rằng bạn được phép dùng bản Kaggle.
CONFIRM_KAGGLE_VSL400_PERMISSION = False  # @param {type:'boolean'}

CLASS_COUNT = 70  # @param {type:'integer'}
# Để trống để chọn tự động; điền gloss_id (tên từ trong metadata) để lấy đúng các từ đó theo thứ tự.
GLOSS_IDS = []
REUSE_FULL_PREPARATION = True  # @param {type:'boolean'}
REUSE_KEYPOINTS = True  # @param {type:'boolean'}
MIN_KEYPOINT_COVERAGE = 0.95  # @param {type:'number'}

# Mô hình theo slide thiết kế: 2 graph block · Spatial 2 layer × 4 head · Temporal 3 layer × 4 head.
RUN_NAME = 'graph2_spatial2x4_temporal3x4_v1'  # @param {type:'string'}
TARGET_FRAMES = 64  # @param {type:'integer'}
GRAPH_DIM = 128  # @param {type:'integer'}
TEMPORAL_DIM = 256  # @param {type:'integer'}
DROPOUT = 0.2  # @param {type:'number'}
EPOCHS = 100  # @param {type:'integer'}
BATCH_SIZE = 32  # @param {type:'integer'}
LEARNING_RATE = 5e-4  # @param {type:'number'}
WEIGHT_DECAY = 0.05  # @param {type:'number'}
LABEL_SMOOTHING = 0.1  # @param {type:'number'}
WARMUP_EPOCHS = 5  # @param {type:'integer'}
PATIENCE = 15  # @param {type:'integer'}
SEED = 42  # @param {type:'integer'}
RESUME = True  # @param {type:'boolean'}
# Tập test chỉ dùng một lần, sau khi đã chốt cấu hình dựa trên validation.
RUN_TEST = False  # @param {type:'boolean'}

GLOSS_IDS = [str(value) for value in GLOSS_IDS]
if not CONFIRM_KAGGLE_VSL400_PERMISSION:
    raise RuntimeError(
        'Bản Kaggle là bản đăng lại của VSL400. Xác nhận quyền sử dụng với tác giả VSL400 '
        'rồi đặt CONFIRM_KAGGLE_VSL400_PERMISSION = True.'
    )
if not GLOSS_IDS and CLASS_COUNT < 2:
    raise ValueError('CLASS_COUNT phải từ 2 trở lên.')
if not 0 < MIN_KEYPOINT_COVERAGE <= 1:
    raise ValueError('MIN_KEYPOINT_COVERAGE phải nằm trong (0, 1].')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path('/content/silent-signal')
METADATA_ROOT = Path('/content/VSL400_metadata')
# Dùng chung với notebook 11 (nguồn Kaggle) để tái sử dụng manifest/split toàn bộ 400 từ.
RESULTS_ROOT = Path('/content/drive/MyDrive/silent-signal-results') / f'vsl400_kaggle_v{KAGGLE_VERSION}'
FULL_MANIFEST = RESULTS_ROOT / 'manifests/vsl400.parquet'
FULL_SPLIT = RESULTS_ROOT / 'splits/vsl400_signer_split.json'
FULL_CONFIG = RESULTS_ROOT / 'configs/vsl400.full.yaml'

SUBSET_NAME = 'custom_' + '_'.join(GLOSS_IDS) if GLOSS_IDS else f'top{CLASS_COUNT}'
SUBSET_ROOT = RESULTS_ROOT / 'subsets' / f'{SUBSET_NAME}_front_mediapipe'
PREPARED_ROOT = SUBSET_ROOT / 'prepared'
MANIFEST = PREPARED_ROOT / 'manifest.csv'
SELECTION = PREPARED_ROOT / 'selection.json'
KEYPOINTS_PATH = SUBSET_ROOT / 'keypoints' / 'mediapipe76_front.npz'
RUN_ROOT = SUBSET_ROOT / 'runs' / RUN_NAME
FIGURES_ROOT = RUN_ROOT / 'figures'

for path in (PREPARED_ROOT, KEYPOINTS_PATH.parent, FIGURES_ROOT, FULL_CONFIG.parent):
    path.mkdir(parents=True, exist_ok=True)
print('Kết quả lưu trên Drive:', SUBSET_ROOT)
print('Lượt train:', RUN_ROOT)

## Mã nguồn và môi trường

Clone đúng nhánh `PROJECT_GIT_REF` rồi cài dự án vào môi trường Colab (dùng PyTorch có sẵn của Colab; không cần OpenMMLab). Token Kaggle được đọc từ Colab Secrets vào biến môi trường của các tiến trình con và **không bao giờ in ra**.

In [ ]:
import os
import subprocess
import sys
import time

def run(command, env=None):
    command = [str(part) for part in command]
    print('+', ' '.join(command), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    print(f'[{(time.perf_counter() - started) / 60:.1f} min]', flush=True)
    if code:
        raise subprocess.CalledProcessError(code, command)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', PROJECT_GIT_REF, PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', '--depth', '1', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', '--detach', 'FETCH_HEAD'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', PROJECT_ROOT])
# A running kernel does not see a fresh editable install until restart; import from src/ directly.
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()

import torch
from google.colab import userdata

print('Project commit:', PROJECT_COMMIT)
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('Cảnh báo: không có GPU — train vẫn chạy nhưng chậm hơn nhiều.')

PROCESS_ENV = os.environ.copy()
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
try:
    PROCESS_ENV['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    PROCESS_ENV['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception as exc:
    raise RuntimeError('Thiếu Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY, hoặc chưa bật Notebook access.') from exc
print('Đã nạp token Kaggle từ Colab Secrets (không in ra).')

def cli(module, *arguments):
    run([sys.executable, '-m', f'silent_signal.cli.{module}', *arguments], env=PROCESS_ENV)

## Bước 1 — Manifest và split theo người ký cho toàn bộ 400 từ

Chỉ đọc 21 file JSON (7 phần × 3 góc) trong file zip Kaggle rồi ghép lại, giống `merge_splits.py` của tác giả. Split được tạo trên **toàn bộ** dataset trước khi chọn từ, nên việc chọn 70 từ không ảnh hưởng tới việc phân người ký. Các lần chạy sau dùng lại kết quả trên Drive (`REUSE_FULL_PREPARATION=True`).

In [ ]:
import json
import yaml

if REUSE_FULL_PREPARATION and FULL_MANIFEST.is_file() and FULL_SPLIT.is_file():
    print('Dùng lại manifest/split đã có:', FULL_MANIFEST)
else:
    cli('fetch_vsl400_kaggle', 'metadata', '--output-root', METADATA_ROOT,
        '--dataset', KAGGLE_DATASET, '--version', KAGGLE_VERSION)
    full_config = yaml.safe_load((PROJECT_ROOT / 'configs/dataset/vsl400.yaml').read_text(encoding='utf-8'))
    full_config['dataset']['root'] = str(METADATA_ROOT)
    full_config['outputs'] = {
        'manifest_csv': str(RESULTS_ROOT / 'manifests/vsl400.csv'),
        'manifest_parquet': str(FULL_MANIFEST),
        'labels': str(RESULTS_ROOT / 'labels/vsl400_labels.json'),
        'split': str(FULL_SPLIT),
        'report': str(RESULTS_ROOT / 'reports/metadata_validation_report.json'),
        'invalid_records': str(RESULTS_ROOT / 'reports/metadata_invalid_records.csv'),
    }
    FULL_CONFIG.write_text(yaml.safe_dump(full_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
    cli('prepare', 'build-manifest', '--config', FULL_CONFIG)
    cli('prepare', 'create-splits', '--config', FULL_CONFIG)

full_split = json.loads(FULL_SPLIT.read_text(encoding='utf-8'))
print('Chia theo người ký (toàn bộ 400 từ, cả 3 góc):')
for name in ('train', 'validation', 'test'):
    print(f"  {name:<10} {full_split['signer_counts'][name]:>2} người ký  "
          f"{full_split['instance_counts'][name]:>6} lần ký  signer={full_split['signer_ids'][name]}")

## Bước 2 — Chọn 70 từ

Xếp các từ theo **số lần ký trong tập train** (validation/test chỉ dùng để yêu cầu từ có mặt ở cả 3 split). Khi bằng nhau thì giữ thứ tự gốc của từ. `class_index` mới chạy từ 0 theo thứ hạng, tên tiếng Việt được giữ nguyên. Danh sách được lưu ở `prepared/selection.json` trên Drive.

In [ ]:
command = ['--manifest', FULL_MANIFEST, '--output-root', PREPARED_ROOT,
           '--classes', CLASS_COUNT, '--dataset-name', 'vsl400']
for gloss_id in GLOSS_IDS:
    command += ['--gloss-id', gloss_id]
cli('select_classes', *command)

selection = json.loads(SELECTION.read_text(encoding='utf-8'))
print(f"\n{selection['class_count']} từ | lần ký theo split: {selection['instances']}")
print(f"{'rank':>4} {'class':>5}  {'từ':<28} {'train':>5} {'val':>4} {'test':>4}")
for item in selection['classes']:
    counts = item['instances']
    print(f"{item['rank']:>4} {item['class_index']:>5}  {item['gloss_name'][:28]:<28} "
          f"{counts['train']:>5} {counts['validation']:>4} {counts['test']:>4}")

## Bước 3 — Tải keypoint MediaPipe của 70 từ (góc front)

Lấy bản **canonical** trong `processed/processed/keypoints_splited` (bỏ qua `processed_augmented`, vì bản đó đã bị nội suy lấp đầy nên mất thông tin điểm thiếu). Mỗi clip được ghép theo mã video **và** tên từ. Clip nào không ghép được sẽ bị bỏ và ghi vào báo cáo, không đoán. Nếu tỉ lệ ghép được thấp hơn `MIN_KEYPOINT_COVERAGE`, cell sẽ dừng.

In [ ]:
from silent_signal.data.keypoint_pack import read_packed_keypoints

if REUSE_KEYPOINTS and KEYPOINTS_PATH.is_file():
    print('Dùng lại keypoint đã đóng gói:', KEYPOINTS_PATH)
else:
    cli('fetch_vsl400_kaggle', 'keypoints', '--manifest', MANIFEST, '--view', 'front',
        '--output', KEYPOINTS_PATH, '--dataset', KAGGLE_DATASET, '--version', KAGGLE_VERSION,
        '--min-coverage', MIN_KEYPOINT_COVERAGE)

packed = read_packed_keypoints(KEYPOINTS_PATH)
print(json.dumps(packed.metadata['counts'], indent=2))
print(f'{len(packed.sample_ids):,} clip | {len(packed.frames):,} frame | '
      f'{KEYPOINTS_PATH.stat().st_size / 1024**2:.0f} MB trên Drive')

## Bước 4 — Xem nhanh dữ liệu

Độ dài clip, tỉ lệ frame bị mất cả thân hoặc cả một bàn tay, và một bộ khung mẫu. Thân người và mỗi bàn tay được người đăng chuẩn hóa **riêng** (mỗi phần về khoảng [-0,5; 0,5]), nên được vẽ thành 3 khung.

In [ ]:
import csv

import matplotlib.pyplot as plt
import numpy as np
from silent_signal.pose.layouts import MEDIAPIPE_UPPER68_V1 as LAYOUT

sources = list(LAYOUT.source_indices)
parts = {part: [i for i, joint in enumerate(LAYOUT.joints) if joint.body_part in part]
         for part in (('body', 'face'), ('left_hand',), ('right_hand',))}
lengths = np.diff(packed.offsets)
lost = {part: [] for part in parts}
for index in range(len(packed.sample_ids)):
    observed = np.any(packed.sequence(index)[:, sources] != 0, axis=-1)
    for part, joints in parts.items():
        lost[part].append(float(1 - observed[:, joints].any(axis=1).mean()))

figure, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(lengths, bins=40, color='#4c72b0')
axes[0].set(title='Số frame mỗi clip (sau khi người đăng cắt đoạn tĩnh)', xlabel='frame', ylabel='clip')
labels = ['thân + mặt', 'tay trái', 'tay phải']
axes[1].bar(labels, [100 * np.mean(values) for values in lost.values()], color=['#55a868', '#c44e52', '#8172b2'])
axes[1].set(title='Tỉ lệ frame bị mất cả phần (%)', ylabel='%')
figure.tight_layout()
figure.savefig(FIGURES_ROOT / 'data_overview.png', dpi=140)
plt.show()
print(f'Số frame: trung vị {np.median(lengths):.0f}, min {lengths.min()}, max {lengths.max()}')

with MANIFEST.open(encoding='utf-8', newline='') as handle:
    manifest_rows = {row['sample_id']: row for row in csv.DictReader(handle)}
sample = next(s for s in packed.sample_ids if manifest_rows[s]['split'] == 'train')
sequence = packed.sequence(packed.index[sample])[:, sources]
frame = sequence[len(sequence) // 2]
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (part, joints), title in zip(axes, parts.items(), labels):
    members = set(joints)
    for a, b in LAYOUT.edges:
        if a in members and b in members and frame[a].any() and frame[b].any():
            axis.plot(frame[[a, b], 0], frame[[a, b], 1], color='#333333', linewidth=1)
    visible = [j for j in joints if frame[j].any()]
    axis.scatter(frame[visible, 0], frame[visible, 1], s=12, color='#dd5555')
    axis.set(title=title, aspect='equal')
    axis.invert_yaxis()
figure.suptitle(f"{manifest_rows[sample]['gloss_name']} — {sample} (frame giữa)")
figure.tight_layout()
figure.savefig(FIGURES_ROOT / 'skeleton_sample.png', dpi=140)
plt.show()

## Bước 5 — Train Graph Encoder + Spatial/Temporal Transformer

Tối ưu bằng AdamW (warmup rồi cosine), AMP fp16 trên GPU và cắt gradient. Mỗi epoch in loss, top-1/top-5 và macro-F1 trên validation. Checkpoint `last_checkpoint.pt` và `best_checkpoint.pt` nằm trên Drive. Nếu Colab ngắt, chạy lại cell này là resume. Nếu đổi siêu tham số, hãy đổi `RUN_NAME`, vì notebook từ chối resume khi thiết lập đã khác.

In [ ]:
arguments = [
    '--manifest', MANIFEST, '--keypoints', KEYPOINTS_PATH, '--output-root', RUN_ROOT,
    '--view', 'front', '--target-frames', TARGET_FRAMES,
    '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--learning-rate', LEARNING_RATE,
    '--weight-decay', WEIGHT_DECAY, '--label-smoothing', LABEL_SMOOTHING,
    '--warmup-epochs', WARMUP_EPOCHS, '--patience', PATIENCE, '--seed', SEED,
    '--graph-dim', GRAPH_DIM, '--graph-blocks', 2, '--spatial-layers', 2, '--spatial-heads', 4,
    '--temporal-dim', TEMPORAL_DIM, '--temporal-layers', 3, '--temporal-heads', 4,
    '--dropout', DROPOUT, '--workers', 2, '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
]
if not RESUME:
    arguments.append('--no-resume')
if RUN_TEST:
    arguments.append('--run-test')
(RUN_ROOT / 'notebook_run.json').write_text(json.dumps({
    'project_commit': PROJECT_COMMIT, 'kaggle_dataset': KAGGLE_DATASET, 'kaggle_version': KAGGLE_VERSION,
    'arguments': [str(argument) for argument in arguments],
}, indent=2) + '\n', encoding='utf-8')
cli('train_pose_transformer', *arguments)

## Bước 6a — Đồ thị train / validation và đánh giá tự động

Chỉ cần `history.json` (được ghi sau **mỗi** epoch), nên có thể xem khi train đang dở: dừng cell train (checkpoint vẫn giữ, chạy lại là resume), chạy cell này, rồi chạy lại cell train.

- **6 biểu đồ:** loss, top-1 (kèm mức ngẫu nhiên 1/70), top-5, macro-F1, khoảng cách train–validation, và learning rate. Đường chấm đứng là epoch có validation loss thấp nhất; checkpoint ở epoch này được dùng để đánh giá.
- Chỉ số **train** được đo trên batch đã tăng cường và dropout đang bật, nên thường **thấp hơn** so với khi đo trên clip train sạch.
- **Đánh giá tự động** dựa trên ngưỡng đơn giản (overfitting, underfitting, còn cải thiện, validation dao động). Hãy đọc cùng với biểu đồ, không thay cho biểu đồ.

In [ ]:
from silent_signal.evaluation.training_curves import load_run, plot_training_curves, summarize_history

history, report = load_run(RUN_ROOT)
summary = summarize_history(history, report=report)
plot_training_curves(history, FIGURES_ROOT / 'training_curves.png', summary=summary, title=RUN_NAME)
plt.show()
(RUN_ROOT / 'training_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')

last_epoch = history[-1]['epoch']
shown = history if len(history) <= 30 else [
    row for row in history if row['epoch'] % 5 == 0 or row['epoch'] in (1, summary['best_epoch'], last_epoch)]
print(f"{'epoch':>5} {'train loss':>10} {'val loss':>9} {'train top1':>10} {'val top1':>9} "
      f"{'val top5':>9} {'val F1':>7} {'lr':>9}")
for row in shown:
    mark = ' *' if row['epoch'] == summary['best_epoch'] else ''
    print(f"{row['epoch']:>5} {row['train_loss']:>10.3f} {row['validation_loss']:>9.3f} "
          f"{row['train_top1']:>10.3f} {row['validation_top1']:>9.3f} {row['validation_top5']:>9.3f} "
          f"{row['validation_macro_f1']:>7.3f} {row['learning_rate']:>9.2e}{mark}")
print('(* = epoch tốt nhất theo validation loss)')

best = summary['best']
state = 'đã dừng sớm' if summary['stopped_early'] else ('đang train dở' if report is None else 'đã chạy hết số epoch')
print(f"\nĐã train {summary['epochs']} epoch ({state}). Epoch tốt nhất: {summary['best_epoch']} — "
      f"val loss {best['validation_loss']:.3f}, top-1 {best['validation_top1']:.1%}, "
      f"top-5 {best['validation_top5']:.1%}, macro-F1 {best['validation_macro_f1']:.1%}.")
if summary['best_top1_epoch'] != summary['best_epoch']:
    print(f"Top-1 validation cao nhất là {summary['best_validation_top1']:.1%} ở epoch {summary['best_top1_epoch']} "
          '(checkpoint vẫn chọn theo validation loss, ổn định hơn với tập validation nhỏ).')
messages = {
    'overfitting': f"Overfitting: {summary['epochs_since_best']} epoch sau epoch tốt nhất, validation loss tăng "
                   f"{summary['validation_loss_rise_since_best']:.0%} trong khi train loss vẫn giảm "
                   f"{summary['train_loss_drop_since_best']:.0%}. Checkpoint tốt nhất đã được giữ; lượt sau có thể "
                   'tăng DROPOUT/WEIGHT_DECAY, giảm GRAPH_DIM/TEMPORAL_DIM hoặc giảm PATIENCE.',
    'large_generalization_gap': f"Khoảng cách train–validation lớn: top-1 chênh {summary['top1_gap_at_best']:.0%} ở epoch "
                                'tốt nhất — mô hình khớp người ký trong train tốt hơn nhiều so với người ký mới.',
    'underfitting': f"Underfitting: top-1 trên train chỉ {best['train_top1']:.0%} — mô hình chưa học được; tăng EPOCHS, "
                    'LEARNING_RATE hoặc kích thước mô hình, và kiểm tra lại dữ liệu.',
    'still_improving': 'Validation vẫn đang cải thiện ở epoch cuối — có thể tăng EPOCHS (đổi RUN_NAME để train lượt mới).',
    'noisy_validation': f"Validation dao động mạnh (trung bình {summary['validation_top1_fluctuation']:.1%} mỗi epoch "
                        'ngoài xu hướng): tập validation chỉ có 3 người ký, nên xem xu hướng thay vì một epoch lẻ.',
    'no_clear_issue': 'Không thấy dấu hiệu bất thường rõ ràng.',
}
print('\nĐánh giá tự động:')
for finding in summary['findings']:
    print(' -', messages[finding])

## Bước 6b — Kết quả chi tiết (sau khi train xong)

Chỉ số của checkpoint tốt nhất, ma trận nhầm lẫn (chuẩn hóa theo hàng), F1 của từng từ và các cặp từ hay bị nhầm. Hình được lưu ở `figures/` trong thư mục lượt train.

In [ ]:
report = json.loads((RUN_ROOT / 'report.json').read_text(encoding='utf-8'))
names = report['data']['class_names']
print(json.dumps({key: report[key] for key in ('parameters', 'epochs_completed', 'best_epoch',
                                                'best_validation_loss', 'stopped_early', 'evaluation')},
                 ensure_ascii=False, indent=2))

for split in [s for s in ('validation', 'test') if (RUN_ROOT / f'confusion_{s}.csv').is_file()]:
    confusion = np.loadtxt(RUN_ROOT / f'confusion_{split}.csv', delimiter=',', dtype=np.int64)
    normalized = confusion / np.maximum(confusion.sum(axis=1, keepdims=True), 1)
    figure, axis = plt.subplots(figsize=(18, 16))
    image = axis.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    axis.set_xticks(range(len(names)), names, rotation=90, fontsize=7)
    axis.set_yticks(range(len(names)), names, fontsize=7)
    axis.set(title=f'Ma trận nhầm lẫn ({split}, chuẩn hóa theo hàng)', xlabel='dự đoán', ylabel='đúng')
    figure.colorbar(image, fraction=0.03)
    figure.tight_layout()
    figure.savefig(FIGURES_ROOT / f'confusion_{split}.png', dpi=140)
    plt.show()

    with (RUN_ROOT / f'per_class_{split}.csv').open(encoding='utf-8') as handle:
        per_class = sorted(csv.DictReader(handle), key=lambda row: float(row['f1']))
    figure, axis = plt.subplots(figsize=(10, 16))
    axis.barh([row['gloss'] for row in per_class], [float(row['f1']) for row in per_class], color='#4c72b0')
    axis.set(title=f'F1 từng từ ({split})', xlim=(0, 1))
    axis.tick_params(axis='y', labelsize=7)
    figure.tight_layout()
    figure.savefig(FIGURES_ROOT / f'per_class_f1_{split}.png', dpi=140)
    plt.show()

    pairs = [(confusion[i, j], names[i], names[j]) for i in range(len(names)) for j in range(len(names))
             if i != j and confusion[i, j] > 0]
    print(f'\nCác cặp hay nhầm nhất ({split}): đúng → dự đoán (số clip)')
    for count, truth, predicted in sorted(pairs, reverse=True)[:15]:
        print(f'  {truth} → {predicted}: {count}')

## Kết quả trên Drive

`MyDrive/silent-signal-results/vsl400_kaggle_v8/`
- `manifests/`, `splits/vsl400_signer_split.json`: toàn bộ 400 từ, dùng chung với notebook 11.
- `subsets/top70_front_mediapipe/prepared/`: `selection.json` (**danh sách 70 từ**, số lần ký mỗi split, SHA-256 của nguồn), `manifest.csv|parquet`, `labels.json`.
- `subsets/top70_front_mediapipe/keypoints/mediapipe76_front.npz`: keypoint gốc `[T, 76, 3]` của các clip, kèm file nguồn trong zip và danh sách clip không ghép được.
- `subsets/top70_front_mediapipe/runs/<RUN_NAME>/`:
  - `config.json`, `notebook_run.json`: cấu hình, mã commit, SHA-256 của dữ liệu;
  - `history.json`, `last_checkpoint.pt`, `best_checkpoint.pt`;
  - `report.json`, `predictions_*.csv`, `per_class_*.csv`, `confusion_*.csv`;
  - `figures/*.png`.

**Bước tiếp theo:** thêm nhánh RGB (VideoMAE V2 trên video front 224×224 có sẵn trong bộ Kaggle) và ghép bằng cross-attention với `PoseGraphTransformer.encode()`, theo thiết kế RGB–pose.